### BiGG Models

In [1]:
import requests
import pandas as pd
from time import sleep
from Bio import Entrez

Entrez.email = "lyachinas@gmail.com"

# Request for individual model
def get_model_details(model_id):
    url = f"http://bigg.ucsd.edu/api/v2/models/{model_id}"
    r = requests.get(url)
    return r.json()

# Function to get Taxonomy ID from RefSeq genome accession
def get_taxonomy_id_from_genome(genome_accession, organism_name):
    # Try with genome accession first
    try:
        handle = Entrez.esearch(db="nuccore", term=genome_accession)
        record = Entrez.read(handle)
        if record["IdList"]:
            nuccore_id = record["IdList"][0]
            try:
                link_handle = Entrez.elink(dbfrom="nuccore", db="taxonomy", id=nuccore_id)
                link_record = Entrez.read(link_handle)
                taxid = link_record[0]["LinkSetDb"][0]["Link"][0]["Id"]
                return taxid
            except (IndexError, KeyError):
                print(f"No taxonomy link found for {genome_accession}, will try organism name...")
    except Exception as e:
        print(f"Genome accession lookup failed for {genome_accession}: {e}")

    # Fallback: try with organism name
    try:
        handle = Entrez.esearch(db="taxonomy", term=organism_name)
        record = Entrez.read(handle)
        if record["IdList"]:
            taxid = record["IdList"][0]
            print(f"Taxonomy ID found for {organism_name}: {taxid}")
            return taxid
        else:
            print(f"No taxonomy ID found for organism: {organism_name}")
    except Exception as e:
        print(f"Organism name lookup failed for {organism_name}: {e}")

    return None


In [2]:
# Step 1: Get BiGG models
url = 'http://bigg.ucsd.edu/api/v2/models'
response = requests.get(url)
models = response.json()["results"]
print(f'{len(models)} models found')

108 models found


In [3]:
# Step 2: Iterate through models and enrich with genome and taxonomy info
data = []

for model in models:
    model_id = model["bigg_id"]
    organism = model["organism"]

    details = get_model_details(model_id)
    genome_id = details.get("genome_name", "")
    organism = details.get("organism", "")
    taxid = get_taxonomy_id_from_genome(genome_id, organism) if genome_id else None
    print(f'{organism}, {genome_id}, {taxid}')

    data.append({
        "Model Name": model_id,
        "Organism": organism,
        "Genome ID": genome_id,
        "Taxonomy ID": taxid
    })

    sleep(0.5) 

Escherichia coli str. K-12 substr. MG1655, NC_000913.3, 511145
Taxonomy ID found for Homo sapiens: 9606
Homo sapiens, GCF_000001405.33, 9606
Escherichia coli str. K-12 substr. MG1655, NC_000913.3, 511145
Escherichia coli str. K-12 substr. MG1655, NC_000913.3, 511145
No taxonomy link found for CP000099.1, will try organism name...
Taxonomy ID found for Methanosarcina barkeri str. Fusaro: 269797
Methanosarcina barkeri str. Fusaro, CP000099.1, 269797
Geobacter metallireducens GS-15, CP000148.1, 269799
No taxonomy link found for NC_036159.1, will try organism name...
Taxonomy ID found for Plasmodium berghei: 5821
Plasmodium berghei, NC_036159.1, 5821
Plasmodium cynomolgi strain B, NC_020396.1, 1120755
Plasmodium falciparum 3D7, NC_004325.2, 36329
No taxonomy link found for NC_011902.1, will try organism name...
Taxonomy ID found for Plasmodium knowlesi strain H: 5851
Plasmodium knowlesi strain H, NC_011902.1, 5851
Plasmodium vivax Sal-1, NC_009906.1, 5855
Escherichia coli APEC O1, NC_00856

In [4]:
# Step 3: Export as DataFrame

df_bigg_models = pd.DataFrame(data)
print(f'Total BiGG models: {len(df_bigg_models)}')


df_bigg_models_unique = df_bigg_models.drop_duplicates(subset=["Genome ID"], keep="first")
print(f'Unique BiGG models: {len(df_bigg_models_unique)}')

# Drop entries with missing Taxonomy ID
df_bigg_models_clean = df_bigg_models_unique[df_bigg_models_unique["Taxonomy ID"].notna()]
print(f'BiGG models with found Taxonomy ID: {len(df_bigg_models_clean)}')


Total BiGG models: 108
Unique BiGG models: 90
BiGG models with found Taxonomy ID: 88


In [46]:
# csv checkpoint
#df_bigg_models.to_csv("data_pre_db/df_bigg_models.csv", index=False)


### Paxdb 

In [42]:
import zipfile

# Path to the zip file
zip_file_path = 'paxdb-abundance-files-v5.0.zip'

# Open the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # List all files and directories in the zip file
    all_files = zip_ref.namelist()
    
    # Extract the subfolder names (taxonomy IDs)
    taxonomy_ids_in_zip = {name.split('/')[1] for name in all_files if name.count('/') > 1}
    taxonomy_ids_in_zip_unique = {tax_id: None for tax_id in taxonomy_ids_in_zip}
    print(f'Amount of organisms in paxdb: {len(taxonomy_ids_in_zip)}')
    print(f'Amount of unique organisms in paxdb: {len(taxonomy_ids_in_zip_unique)}')

# Extract taxonomy IDs from the DataFrame
taxonomy_ids_in_df = set(df_bigg_models_clean["Taxonomy ID"].astype(str))

# Compare the two sets
missing_in_zip = taxonomy_ids_in_df - taxonomy_ids_in_zip
missing_in_df = taxonomy_ids_in_zip - taxonomy_ids_in_df
matches = taxonomy_ids_in_df & taxonomy_ids_in_zip

# Print the results
print(f'Amount of matches: {len(matches)}')
print(f"Taxonomy IDs in DataFrame but not in zip: {len(missing_in_zip)}")
print(f"Taxonomy IDs in zip but not in DataFrame: {len(missing_in_df)}")

Amount of organisms in paxdb: 170
Amount of unique organisms in paxdb: 170
Amount of matches: 16
Taxonomy IDs in DataFrame but not in zip: 66
Taxonomy IDs in zip but not in DataFrame: 154


In [47]:
BiGG_Paxdb_matches = df_bigg_models_clean[df_bigg_models_clean["Taxonomy ID"].isin(matches)].copy()
BiGG_Paxdb_matches = BiGG_Paxdb_matches.drop_duplicates(subset=["Genome ID"], keep="first")
# csv checkpoint
BiGG_Paxdb_matches.to_csv("data_pre_db/BiGG_Paxdb_matches.csv", index=False)


## 